In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import importlib
import src.features
import src.quality_checks
import src.pipeline

importlib.reload(src.features)
importlib.reload(src.quality_checks)
importlib.reload(src.pipeline)

from src.pipeline import run_pipeline
from src.db import run_query

outputs = run_pipeline(refresh_data=False)

Step 1/6 — Creating schemas...
Step 2/6 — Reusing existing raw parquet files...
Step 3/6 — Loading raw parquet files into DuckDB...
Step 4/6 — Cleaning data, running quality checks, and building model features...
Running quality gate on cleaned historical data...
Quality gate passed.
Step 5/6 — Training model and building final 28-day forecast...
Step 6/6 — Pipeline completed.
Model feature rows: 10925
Final forecast rows: 140


In [2]:
run_query("""
SELECT table_schema, table_name
FROM information_schema.tables
ORDER BY table_schema, table_name
""")

,table_schema,table_name
0,analytics,final_28d_forecast
1,analytics,future_28d_forecast
2,analytics,model_features
3,raw,forecast
4,raw,historical


In [3]:
run_query("""
SELECT *
FROM analytics.final_28d_forecast
ORDER BY city, target_time
LIMIT 35
""")

,city,origin_time,forecast_horizon,target_time,source,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean
0,Baku,2026-04-27,1,2026-04-28,api_forecast,14.100000,0.000000,45.200000,52.000000,90.000000
1,Baku,2026-04-27,2,2026-04-29,api_forecast,14.300000,0.000000,22.100000,63.000000,96.000000
2,Baku,2026-04-27,3,2026-04-30,api_forecast,17.700000,0.000000,19.100000,59.000000,42.000000
3,Baku,2026-04-27,4,2026-05-01,api_forecast,18.000000,0.000000,31.000000,76.000000,20.000000
4,Baku,2026-04-27,5,2026-05-02,api_forecast,19.200000,0.000000,32.300000,80.000000,63.000000
5,Baku,2026-04-27,6,2026-05-03,api_forecast,21.700000,0.000000,20.200000,71.000000,57.000000
6,Baku,2026-04-27,7,2026-05-04,api_forecast,24.200000,0.000000,54.000000,66.000000,84.000000
7,Baku,2026-05-04,8,2026-05-05,ml_model,23.990902,1.444446,27.264306,66.676547,34.993210
8,Baku,2026-05-04,9,2026-05-06,ml_model,23.951948,1.477447,27.264306,66.676547,34.993210
9,Baku,2026-05-04,10,2026-05-07,ml_model,23.951948,1.477447,27.264306,66.676547,34.993210


In [4]:
run_query("""
SELECT city, source, COUNT(*) AS rows
FROM analytics.final_28d_forecast
GROUP BY city, source
ORDER BY city, source
""")

,city,source,rows
0,Baku,api_forecast,7
1,Baku,ml_model,21
2,Gabala,api_forecast,7
3,Gabala,ml_model,21
4,Guba,api_forecast,7
5,Guba,ml_model,21
6,Lankaran,api_forecast,7
7,Lankaran,ml_model,21
8,Shaki,api_forecast,7
9,Shaki,ml_model,21
